In [1]:
import glob
files = glob.glob("data/nifty*.parquet")

In [2]:
from datetime import time

START_TIME = time(9,17)
EXIT_TIME = time(15,16)
REENTRY_END = time(9,34)

In [3]:
import pandas as pd
df = pd.read_parquet(files[-1])

In [4]:
pivot = df.pivot_table(
    index = ['datetime','strike_price'],
    columns='right',
    values='close_opt',
    aggfunc='last'
).reset_index()

In [5]:
pivot["straddle"] = pivot["Call"] + pivot["Put"]

In [6]:
OFF_SET_TH = 0.2

In [10]:

for i,row in df.groupby('datetime'):
    if i.time()< START_TIME:
        spot_close = row['close_spot'].iloc[0]
        atm_strike = round(spot_close/50)*50
        straddle_row = pivot[(pivot['datetime']==i) & (pivot['strike_price']==atm_strike)]

        if straddle_row.empty:
            continue

        straddle_p = straddle_row['straddle'].iloc[0]
        off_set = OFF_SET_TH * straddle_p

        call_row = row[
            (row["right"]=="Call") &
            (row["strike_price"] > atm_strike-50) &
            (row["close_opt"] <= off_set)
        ].sort_values("strike_price").head(1)

        put_row = row[
            (row["right"]=="Put") &
            (row["strike_price"] < atm_strike+50) &
            (row["close_opt"] <= off_set)
        ].sort_values("strike_price",ascending=False).head(1)

        if call_row.empty or put_row.empty:
            continue
    break            
       


In [11]:
call_strike = call_row["strike_price"].iloc[0]
print(call_strike)
put_strike = put_row["strike_price"].iloc[0]
print(put_strike)

25700
25450


In [12]:
def select_strike(row,atm,offset):
    call_row = row[
            (row["right"] == "Call") &
            (row["strike_price"] > atm - 50) &
            (row["close_opt"] <= offset)
        ].sort_values("strike_price").head(1)

    put_row = row[
            (row["right"] == "Put") &
            (row["strike_price"] < atm + 50) &
            (row["close_opt"] <= offset)
        ].sort_values("strike_price", ascending=False).head(1)

    return call_row, put_row


In [13]:
call_data, put_data = select_strike(row,atm_strike,off_set)

In [14]:
call_strike = call_data["strike_price"].iloc[0]
put_strike = put_data["strike_price"].iloc[0]

In [15]:
call_strike

np.int64(25700)

In [16]:
put_strike

np.int64(25450)